# ch05 Bonus 08：扩展词表（Extending Tokenizers）

> 对照官方 `ch05/09_extending-tokenizers`

## 一句话

GPT-2 的 BPE 词表（50257）不擅长中文、代码等。本 notebook 演示如何往现有 tokenizer **添加特殊 token / 新词**，并同步扩展模型的 embedding 矩阵。

## 为什么需要扩展

- **中文**：GPT-2 把每个汉字切成多个 byte token，序列被拉长、效率低
- **特殊任务标记**：如对话的 `<|user|>` `<|assistant|>`，或分类的 `[CLS]`
- **领域词**：医学、法律的专业术语

## 两步走

1. **扩展 tokenizer**：加入新 token（需保存新 vocab）
2. **扩展模型 embedding**：`nn.Embedding` 和输出层的行数要对应增加

> 关键：新加的 token 是「未训练」的，需要继续预训练或微调让模型学会它们。

In [ ]:
import tiktoken
import torch
import torch.nn as nn
from src.gpt import GPTModel, GPT_CONFIG_124M

tok = tiktoken.get_encoding("gpt2")
print(f"原词表大小: {tok.n_vocab}")

# 观察 GPT-2 对中文的编码（低效：一个字变成多个 token）
zh = "你好世界"
ids = tok.encode(zh)
print(f"\n'{zh}' 编码: {ids}（{len(ids)} 个 token，{len(zh)} 个字）")
print("💡 每个汉字被切成多个 byte token，序列变长、信息密度低。")

In [ ]:
# 演示扩展模型的 embedding 以容纳新 token
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4})  # 小配置 demo

torch.manual_seed(123)
model = GPTModel(cfg)
old_vocab = cfg["vocab_size"]
print(f"原 vocab_size: {old_vocab}")
print(f"原 token embedding: {tuple(model.tok_emb.weight.shape)}")
print(f"原 output head:     {tuple(model.out_head.weight.shape)}")

# 假设新增 3 个特殊 token
n_new = 3
new_vocab = old_vocab + n_new

def extend_embedding(emb, n_new):
    """扩展 embedding 矩阵：在末尾添加 n_new 行。"""
    old = emb.weight.data
    # 新 token 的 embedding 用均值初始化（比全零更合理）
    new_rows = old.mean(dim=0, keepdim=True).repeat(n_new, 1)
    new_rows += 0.02 * torch.randn_like(new_rows)  # 加点噪声
    new_weight = torch.cat([old, new_rows], dim=0)
    emb.weight = nn.Parameter(new_weight)

extend_embedding(model.tok_emb, n_new)
extend_embedding(model.out_head, n_new)

print(f"\n扩展后 token embedding: {tuple(model.tok_emb.weight.shape)}")
print(f"扩展后 output head:     {tuple(model.out_head.weight.shape)}")
print(f"✓ 模型现在能接受 vocab id 高达 {new_vocab - 1} 的输入")

# 验证新 token 能前向
idx = torch.tensor([[old_vocab, old_vocab + 1, old_vocab + 2]])  # 用新 token id
model.eval()
with torch.no_grad():
    out = model(idx)
print(f"新 token 前向输出: {tuple(out.shape)} ✓")
print("\n💡 新 token 的 embedding 是初始化值，需继续训练让模型学会它们的含义。")